# 1. Library packages

In [1]:
setwd("/home/liyanguo/MyImmuCell/")

In [2]:
source("00_code_MyImmuCell/0_data_preperation/Environment.R")

# 2. R+python: Read, QC, GeneFull_Ex50pAS_raw 

In [3]:
#注意更改文件夹
sample_list = list.files("01_rawdata/MyImmuCell_scRNA/")
head(sample_list)
length(sample_list)

[1] "D0500_Rep1" "D0500_Rep2" "D0501_Rep1" "D0501_Rep2" "D0502_Rep1"
[6] "D0502_Rep2"

[1] 972

In [24]:
if(T){
  #提前算好核心数填进来
  sfInit(parallel = T,cpus = 60)
  sfLibrary(Seurat)
  sfLibrary(scCustomize)
  sfLibrary(anndata)
  sfLibrary(dplyr)
  sfLibrary(snowfall)
  sfLibrary(reticulate)
  sfLibrary(Azimuth)
  sfLibrary(data.table)
  sfLibrary(tidyverse)
  sfLapply(sample_list,function(i){
    print(paste0("Process sample ID: ", i),sep = "\n")


      
    #cite-seq rna raw data
    filedir = paste0("01_rawdata/MyImmuCell_scRNA/",i,"/GeneFull_Ex50pAS/raw/")
    counts = Read10X(filedir)
    #cite-seq adt raw data
    adt_counts = fread(paste0("01_rawdata/MyImmuCell_ADT/",i,"_ADT.csv"))
    adt_counts = column_to_rownames(adt_counts,var="V1")
    stopifnot("Check cell barcodes!"= all(colnames(adt_counts) == colnames(counts)))





      
    #adt raw h5ad，for dsb normalization
    adt_object <- CreateSeuratObject(
      counts = adt_counts,
      project = i,
      min.cells = 0,
      min.features = 0,
      assay = "ADT")
    #Add SampleID and DonorID
    adt_object = RenameCells(object = adt_object, add.cell.id = i)
    adt_object$SampleID = i
    adt_object$DonorID = limma::strsplit2(adt_object$SampleID,"_")[,1]
      
    #To anndata
    adt_adata <- AnnData(
      X = t(LayerData(adt_object, "counts")),
      obs = adt_object@meta.data
    )
    write_h5ad(adt_adata, paste0("02_Read_QC/MyImmuCell_ADT_h5ad/",i,"_ADT_raw.h5ad"),compression="gzip")



      
      
    #cellbender results: remove emptydrops
    cellbender = read.csv(paste0("01_rawdata/MyImmuCell_scRNA/",i,"/outs/filtered/barcodes.tsv.gz"),header=F)
    counts = counts[,cellbender$V1]
    adt_counts = adt_counts[,cellbender$V1]
      
    stopifnot("Check cell barcodes!"= all(colnames(counts) == colnames(adt_counts)))



      
    object <- CreateSeuratObject(
      counts = counts,
      min.cells = 0,
      assay = "RNA",
      min.features = 0,
      project = i)

    if ("nFeature_RNA" %in% colnames(object@meta.data) & "nCount_RNA" %in% colnames(object@meta.data)) {
      print("exist")
    }else{
      object[["nCount_RNA"]] <- colSums(LayerData(object, "counts"))
      object[["nFeature_RNA"]] <- colSums(LayerData(object, "counts") > 0)
    }
    
    #Add SampleID and DonorID
    object = RenameCells(object = object, add.cell.id = i)
    object$SampleID = i
    object$DonorID = limma::strsplit2(object$SampleID,"_")[,1]
      
    #写入质控信息
    object <- Add_Cell_QC_Metrics(object, species = "human",
                                  add_complexity = TRUE,
                                  add_top_pct = TRUE,
                                  add_IEG = TRUE,
                                  add_MSigDB = TRUE,
                                  add_cell_cycle = TRUE,
                                  add_mito_ribo = TRUE,
                                  add_hemo = TRUE,
                                  overwrite = TRUE)

    #celltypist 区分Neu和PBMC
    source("00_code_MyImmuCell/0_data_preperation/celltypist.R")
    pred = celltypist_prediction(object)
    object = AddMetaData(object,metadata = pred)

    #针对nCount_RNA均值低于2000的群体
    object_low_nCount_RNA=subset(object,subset=Reference_Atlas_L1L2_pl %in% c("CEACAM8- Neutrophil",'Basophil','Platelet'))
    #针对nCount_RNA均值高于2000的群体，主要是PBMC
    object_high_nCount_RNA=subset(object,
                                  cells=setdiff(colnames(object),colnames(object_low_nCount_RNA))
                                 )

    #标准化和降维
    source("00_code_MyImmuCell/0_data_preperation/NormalizeData2Umap.R")
    object_low_nCount_RNA=NormalizeData2Umap(object_low_nCount_RNA)
    object_high_nCount_RNA=NormalizeData2Umap(object_high_nCount_RNA)

    #Azimuth
    source("00_code_MyImmuCell/0_data_preperation/MyRunAzimuth.R")
    object_low_nCount_RNA <- MyRunAzimuth(object_low_nCount_RNA,
                       reference = "07_ref_model/Azimuth/pbmc_ref/",
                       verbose = F)
    object_high_nCount_RNA <- MyRunAzimuth(object_high_nCount_RNA,
                       reference = "07_ref_model/Azimuth/pbmc_ref/",
                       verbose = F)
    object_low_nCount_RNA <- MyRunAzimuth(object_low_nCount_RNA,
                       reference = "07_ref_model/Azimuth/Human_BMMC/",
                       verbose = F)
    object_high_nCount_RNA <- MyRunAzimuth(object_high_nCount_RNA,
                       reference = "07_ref_model/Azimuth/Human_BMMC/",
                       verbose = F)

    #Doublet detection, doublets will removed during cell type annotation when cell with high level of expression of more than one cell population-specific markers (genes or proteins).
    source("00_code_MyImmuCell/0_data_preperation/Doublet_detection.R")
    object_low_nCount_RNA = Doublet_detection(object_low_nCount_RNA)
    object_high_nCount_RNA = Doublet_detection(object_high_nCount_RNA)

    #出图：每个样本的双细胞分布、细胞分类、中性粒标志物、样本QC
    source("00_code_MyImmuCell/0_data_preperation/single_donor_QC.R")
    single_donor_QC(object_low_nCount_RNA,type="low_nCount_RNA")
    single_donor_QC(object_high_nCount_RNA,type="high_nCount_RNA")

    #To anndata
    adata_low_nCount_RNA <- AnnData(
      X = t(LayerData(object_low_nCount_RNA, "counts")),
      obs = object_low_nCount_RNA@meta.data
    )
    adata_high_nCount_RNA <- AnnData(
      X = t(LayerData(object_high_nCount_RNA, "counts")),
      obs = object_high_nCount_RNA@meta.data
    )
    write_h5ad(adata_low_nCount_RNA, paste0("02_Read_QC/MyImmuCell_RNA_h5ad/",i,"_low_nCount_RNA.h5ad"),compression="gzip")
    write_h5ad(adata_high_nCount_RNA, paste0("02_Read_QC/MyImmuCell_RNA_h5ad/",i,"_high_nCount_RNA.h5ad"),compression="gzip")





      
    # 读取和保存ADT数据,不需要拆Neu和PBMC，下游QC再拆分
    adt_object <- CreateSeuratObject(
      counts = adt_counts,
      project = i,
      min.cells = 0,
      min.features = 0,
      assay = "ADT")
    #Add SampleID and DonorID
    adt_object = RenameCells(object = adt_object, add.cell.id = i)
    adt_object$SampleID = i
    adt_object$DonorID = limma::strsplit2(adt_object$SampleID,"_")[,1]
    
    #To anndata
    adt_adata <- AnnData(
      X = t(LayerData(adt_object, "counts")),
      obs = adt_object@meta.data
    )
    
    write_h5ad(adt_adata, paste0("02_Read_QC/MyImmuCell_ADT_h5ad/",i,"_ADT_filtered.h5ad"),compression="gzip")
  })
  sfStop()
}

Explicit sfStop() is missing: stop now.


Stopping cluster


snowfall 1.84-6.3 initialized (using snow 0.4-4): parallel execution on 60 CPUs.




Library Seurat loaded.


Library Seurat loaded in cluster.




Library scCustomize loaded.


Library scCustomize loaded in cluster.




...

Library Azimuth loaded in cluster.




Library data.table loaded.


Library data.table loaded in cluster.




Library tidyverse loaded.


Library tidyverse loaded in cluster.



Stopping cluster




In [ ]:
# package version
sessionInfo()

#Cell information from Reference Atlas
Celltype_L1_L2_Refine	nCount_RNA mean
B	2571.30652
Basophil	1558.697004
CD4+ T	2644.401668
CD8+ T	2819.485758
CEACAM8+ Neutrophil	4155.751976
CEACAM8- Neutrophil	1716.901049
Classical monocyte	5580.018271
Dendritic	7260.503553
HPSC	5450.444444
MAIT	2786.379136
Mast	8248.431034
NK	2469.633283
Non-classical monocyte	6242.406338
Plamsa	3310.957655
Platelet	1873.940286
Proliferative T/NK	4639.081162
iNKT	3692.97619
pDC	5879.383755
γδ T	2612.79311

#Cell information from Reference Atlas
Celltype_L1_L2_Refine	nFeature_RNA mean
B	1182.152374
Basophil	938.6297343
CD4+ T	1236.86699
CD8+ T	1356.681594
CEACAM8+ Neutrophil	1565.894225
CEACAM8- Neutrophil	852.7107333
Classical monocyte	2075.78506
Dendritic	2471.4073
HPSC	2242.873563
MAIT	1311.301118
Mast	2988.965517
NK	1314.892622
Non-classical monocyte	2242.016803
Plamsa	1064.590119
Platelet	808.6947014
Proliferative T/NK	2052.032465
iNKT	1600.809524
pDC	2387.084167
γδ T	1291.930906